In [1]:
import config
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

C:\Users\moham\anaconda3\envs\vector_database_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\moham\anaconda3\envs\vector_database_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
pc=Pinecone(api_key=config.pinecone_api_key)

In [ ]:
pc.list_indexes()

In [ ]:
for index in pc.list_indexes():
    if index.name == "my-index":   
        pc.delete_index(index.name)


In [ ]:
pc.list_indexes()

In [ ]:
pc.create_index(
    name="my-index",
    dimension = 3,
    metric = "cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [ ]:
pc.list_indexes()

### After you're done, next we'll upload data to our newly created vector databases.

In [ ]:
pc.create_index(
    name="my-index-2",
    dimension = 1536,
    metric = "Euclidean",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [3]:
pc.list_indexes()

IndexList([<name='my-index-2', dim=1536, ready=True>, <name='my-index', dim=3, ready=True>])

### We've learned how to connect to the pinecone server via API and manage indexes in Python.
### Now it's time for the more crucial phase populating the databases with our data.
### This process is called UPSERTING, a term that combines update and insert into a portmanteau.

If a specified vector already exists in our store, or inserting new rows if the vector doesn't exist

in our pinecone database.



## Okay, first we must create an index object to specify which index will be using.

In [4]:
index =pc.Index(name="my-index")

Now the simplest syntax for -- our data is to create a list of tuples, each with an ID and

the value of the vector.



In [5]:
print(type(index))


<class 'pinecone.index.Index'>


In [6]:
index.upsert(vectors=[
    ("Dog",[4.,0.,1.]),
    ("Cat",[4.,0.,1.]),
    ("Chicken",[2.,2.,1.]),
    ("Mantis",[6.,2.,3.]),
    ("Elephant",[4.,0.,1.])
])
#In my case I'll be using animal names with three values the number of legs, wings and tails.

UpsertResponse(upserted_count=5)

how to upload a large text dataset and embed it.

To upload to pinecone, I'm using one of hugging faces text datasets, which is used for training algorithms,

but I wanted to show you how the vector solution scale.

In [8]:
fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True, token=config.hf_token)

In [10]:
fw
#It contains an iterable data set with the same columns we've seen on the Hugging Face website. An iterable data set helpful in managing 
#large data sets, allows us to access data through an iterator, like a for loop, without storing the entire data set on our computer, 
#significantly reducing memory usage.



IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

## Now it's time to convert our data into vector form, which means that we must find an embedding algorithm.


In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\moham\anaconda3\envs\vector_database_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\moham\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12581.57it/s]


In [12]:
pc.create_index(
    name="text",
    dimension = model.get_sentence_embedding_dimension(),
    metric = "cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)
#Unlike our previous vector databases, we can't simply write any dimension number we like here. Instead, we must adhere to the number of
# dimensions used by our embedding algorithm. As you know, during the embedding process, all original data is transformed into a vector with 
#n dimensions, and each vector in our data has the same number of dimensions.

C:\Users\moham\AppData\Local\Temp\ipykernel_28732\1115091175.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension(),


IndexModel(name='text', metric='cosine', host='https://text-dao8846.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [13]:
index=pc.Index(name="text")

After that, we're ready to embed our existing data and upload it to pinecone.

First we'll create a list of vectors that we'll upload to pinecone.



In [14]:
#define the number of items you want to process (subset size)
subset_size=10000 # ex: takes only 10000 items

#Iterate over the dataset and prepare data for upserting
vectors_to_upsert = []
for i, item in enumerate(fw):
    if i>= subset_size:
        break

    text = item['text']
    unique_id = str(item['id'])
    language = item['language']

    #create an embedding for the text
    embedding = model.encode(text, show_progress_bar = False).tolist()

    #prepare metadata
    metadata = {'language': language}

    #append the tuple (id,embedding, metadata) to the list
    vectors_to_upsert.append((unique_id, embedding, metadata))

# upsert data to pincione in bathces
batch_size =1000 #adjust based on your env and dataset
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i: i+batch_size]
    index.upsert(vectors=batch)
print("Subset of data upserted to pinecone index")

Subset of data upserted to pinecone index


## above funvtion explation
As you know, this is the accepted way of uploading vector data. Our list is initially empty.

We'll add vector data to it iteratively.

We'll use a for loop and iterate over the items in our data set.

Remember that we're using an iterable data set, but in essence, we're fetching a row of the data with each iteration.

We'll need to get the text itself.

In order to achieve that, we require the text column of the item.

Then we'll need the ID and let's also add the language.

Then we'll encode this text and transform it into an ndarray because this is required for the -later.

At this stage we could add some metadata elements.

For now, I'll simply add the language and you can experiment with loading different parts of the data as metadata later.

Following the metadata, we can simply add our newly created embeddings, ID and metadata to our vector list.

Okay, what's next?

With the for loop done and dusted, we can start up sorting our vectors.

We'll create batches of size 1000, which shows how many items will be processed simultaneously.

When I enter the for loop, I'll iterate through the list, starting with the first element and proceeding in steps of 1000 until the end.

Okay, in the batch variable, we'll store a list with vectors to upsert starting from the current item to the batch size.

Taking that slice of the list, we're using the batches technique to speed up the process of upsetting the data.

You can experiment with different batch sizes and see which is optimal for your computer.

Keep in mind that having too large of a batch size might be overwhelming for your memory, so there is a trade off.

Then finally, we upsert the vectors from the current batch to the database.